# Python. Семинар 3. Множества и словари вблизи


## 1. Равные ключи — это один ключ

Ключи в словаре различаются не по типу и не по написанию, а по паре «хеш и
равенство». Числа разных типов, равные друг другу, дают одинаковый хеш:


In [ ]:
hash(1), hash(1.0), hash(True)

В лекции 1 уже говорилось, что `True` — это единица. Добавим сюда `1.0`, которое
тоже равно единице, и получим три записи об одном и том же ключе:


In [ ]:
{1: "целое", True: "логическое", 1.0: "вещественное"}  # noqa: F601

Осталась одна пара. Значение — от последней записи, а вот **ключ остался от
первой**: словарь не заменяет ключ, если равный ему уже лежит внутри. Проверить
это можно по типу оставшегося ключа:


In [ ]:
d = {1: "целое", True: "логическое"}  # noqa: F601
key = list(d)[0]

key, type(key).__name__

In [ ]:
e = {True: "логическое", 1: "целое"}  # noqa: F601
first = list(e)[0]

first, type(first).__name__

То же самое во множестве: побеждает тот, кто попал туда первым.


In [ ]:
{1, 1.0, True}, {True, 1.0, 1}  # noqa: B033

Пометки `# noqa` выше стоят не случайно: `ruff` считает повтор равных ключей
ошибкой — `F601` для словаря, `B033` для множества. Здесь повтор намеренный,
поэтому проверка отключена точечно.

Вывод: не смешивайте в ключах числа разных типов. Словарь, где встречаются
и `1`, и `True`, работает предсказуемо, но не так, как ожидает читатель кода.


## 2. Порядок множества не случаен, но и не ваш

Множество, в отличие от словаря, не хранит порядок вставки: элементы лежат в
ячейках хеш-таблицы, обход идёт по ячейкам подряд. У небольших целых хеш равен
самому числу, поэтому порядок выглядит отсортированным — это совпадение, а не
гарантия:


In [ ]:
{3, 1, 2}, {10, 20, 30}, {1, 100, 10000}

Со строками совпадения не будет: их хеш зависит от случайного значения, которое
интерпретатор выбирает при запуске, — защита от атак, где подбирают ключи с
одинаковым хешем и превращают словарь в связный список.

Одно и то же множество в трёх отдельных запусках:


In [ ]:
import subprocess
import sys

for _ in range(3):
    print(
        subprocess.run(
            [sys.executable, "-c", "print({'а', 'б', 'в', 'г', 'д'})"],
            capture_output=True,
            text=True,
        ).stdout.strip()
    )

Порядок разный. Поэтому «возьмём первый элемент множества» невоспроизводимо:
сегодня тест проходит, завтра падает. Нужен порядок — задавайте явно через
`sorted`.


In [ ]:
sorted({"г", "а", "в", "б"})

## 3. `nan` — значение, не равное самому себе

В лекции 1 разбиралось, что вещественные числа сравниваются с оговорками. У
`float` есть особое значение «не число», которое получается, например, из
неопределённости. Оно не равно ничему, включая себя:


In [ ]:
nan = float("nan")

nan == nan

Множество ищет элемент по хешу, а затем сравнивает найденное с искомым. Раз
`nan` не равен себе, найтись он не должен — и действительно не находится:


In [ ]:
float("nan") in {float("nan")}

Но стоит положить и искать **тот же самый объект**, как ответ меняется:


In [ ]:
nan in {nan}

Перед сравнением через `==` контейнер проверяет, не тот ли это самый объект.
Для обычных значений это ничего не меняет, для `nan` — меняет всё. Поэтому два
разных `nan` занимают два места, а один и тот же — одно:


In [ ]:
len({float("nan"), float("nan")}), len({nan, nan})

Вывод: `nan` не годится ни в ключи, ни в элементы множества. Проверять на него
нужно функцией `math.isnan`, а не сравнением.


## 4. Операторы множеств строже методов

В лекции `a | b` и `a.union(b)` шли как равнозначные записи. Разница есть:
оператор требует, чтобы справа тоже было множество, а метод принимает любую
последовательность.


In [ ]:
{1, 2}.union([3, 4]), {1, 2}.intersection((2, 5))

Оператором тот же вызов не пройдёт: `{1, 2} | [3, 4]` даёт `TypeError`. Это
защита — `|` определён и для других типов, и молчаливое приведение списка к
множеству скрывало бы опечатки.

Правило: справа список или строка — вызывайте метод, оператор оставьте для
двух множеств. Строка при этом тоже последовательность и разберётся на символы:


In [ ]:
{"а", "б"}.union("вг")

## 5. `get` не отличает «нет ключа» от «значение None»

`get` возвращает `None`, когда ключа нет. Но `None` — обычное значение, и его
могли положить в словарь намеренно:


In [ ]:
settings = {"прокси": None}

settings.get("прокси"), settings.get("шрифт")

Ответы одинаковы, а ситуации разные: в первом случае настройка задана и значит
«прокси не нужен», во втором не задана вовсе. Различает их `in`:


In [ ]:
"прокси" in settings, "шрифт" in settings

Второй способ — передать `get` заведомо невозможное значение по умолчанию и
сравнить с ним. Такой объект называют сторожевым:


In [ ]:
MISSING = object()

settings.get("прокси", MISSING) is MISSING, settings.get("шрифт", MISSING) is MISSING

`object()` создаёт пустой объект, равный только самому себе, — случайно
совпасть с ним значение из словаря не может.


## 6. `setdefault` собирает словарь списков за один проход

Задача «сгруппировать элементы по признаку» решается словарём, в котором
значения — списки. Мешает только то, что первый раз список нужно создать:


In [ ]:
words = ["кот", "пёс", "коза", "як"]

groups: dict[int, list[str]] = {}
for word in words:
    if len(word) not in groups:
        groups[len(word)] = []
    groups[len(word)].append(word)

groups

`setdefault` делает то же самое одной строкой: возвращает значение по ключу, а
если ключа нет — сначала кладёт туда переданное значение по умолчанию.


In [ ]:
groups = {}
for word in words:
    groups.setdefault(len(word), []).append(word)

groups

Пустой список создаётся на **каждой** итерации, даже когда ключ уже есть, — и
тут же выбрасывается. Для списка это дёшево, для дорогого значения по умолчанию
было бы расточительно.

Включением задача не решается: оно строит значение для каждого ключа независимо
и не умеет накапливать. Готовый инструмент, `defaultdict`, — в лекции 10.
